
PAKISTANI POLITICIAN IMAGE CLASSIFIER — COMPLETE TRAINING PIPELINE
===================================================================
100% STANDALONE — INCLUDES EVERYTHING:
   ✓ Data Loading (from Kaggle input or local directories)
   ✓ Data Merging (combines multiple raw sources with dedup‑safe naming)
   ✓ Face Alignment (MTCNN + landmark‑based eye alignment)
   ✓ Duplicate Removal (pHash near‑duplicate detection)
   ✓ Data Splitting (Stratified Train 75% / Val 15% / Test 10%)
   ✓ Offline Data Augmentation (Albumentations pipeline, 3× multiplier)
   ✓ Model Training (ArcFace + standard classification, 3 CNN backbones)
   ✓ Evaluation & Results (TTA, confusion matrix, per‑class metrics)

INSTRUCTIONS:
   1. Upload this file to Kaggle
   2. Enable GPU (Settings → Accelerator → GPU)
   3. Add the raw‑image dataset in “/kaggle/input/” (Bing + Google/DuckDuckGo)
   4. Run all cells top‑to‑bottom
   5. Download results from /kaggle/working/

ESTIMATED TIME: 2-3 hours (with GPU)

OUTPUT:
   - /kaggle/working/models/*.pth (trained model weights)
   - /kaggle/working/plots/*.png (training curves, confusion matrices)
   - /kaggle/working/results/*.csv (evaluation reports)

Author: M.Hanzala
Version: 2.1 — Production‑ready, scraper‑removed
"""

## Notebook Title and Description
This notebook builds an end-to-end face-classification pipeline for Pakistani politicians using ArcFace-based metric learning and transfer learning backbones.
The workflow is designed for reproducible training and review: data merge, face alignment, deduplication, stratified split, augmentation, model training, and evaluation.

In [1]:
!pip install --no-deps facenet-pytorch imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.6 MB/s eta 0:00:0000:010:01


In [ ]:
# ============================================================
# MODULAR PIPELINE WRAPPER
# All training logic lives in the `training/` package.
# This notebook is a thin orchestration shim.
# ============================================================

import sys, os

# Inject package path for Kaggle environments
for _candidate in [
    '/kaggle/input',
    '/kaggle/working',
    os.path.dirname(os.path.dirname(os.path.abspath('__file__'))),
]:
    if os.path.isdir(os.path.join(_candidate, 'training')) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

from training.config import config  # noqa – confirms package is importable
from training.main   import main

main()



SECTION 1: SETUP & IMPORTS
============================================================================

============================================================================
SECTION 2: CONFIGURATION
============================================================================

## Loss Function: ArcFace
ArcFace is used to optimize angular separation between classes by applying a margin in embedding space.
Key hyperparameters in this notebook are margin m = 0.3 and scale s = 64.0, which balance class separability and optimization stability.


SECTION 2.5: DATA  
============================================================================

## Data Preparation
This stage consolidates raw images from multiple sources, applies face alignment, removes near-duplicates, and prepares a clean split-ready dataset.
Processing order is: merge raw folders -> align faces -> deduplicate -> stratified train/val/test split.

## Data Loader
Load raw image dataset from Kaggle input directory and prepare it for processing.
The loader copies all class folders from `/kaggle/input/politician-faces` (or equivalent) into the local `data/raw` directory for subsequent alignment, deduplication, and splitting.

In [30]:
%pip install --upgrade torch torchvision --no-deps  # or simply: !pip install torch torchvision --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 97.1 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.25.0+cu128
    Uninstalling torchvision-0.25.0+cu128:
      Successfully uninstalled torchvision-0.25.0+cu128
  Attempting uninstall: torch
    Found existing installation: torch 2.10.0+cu128
    Uninstalling torch-2.10.0+cu128:
      Successfully uninstalled torch-2.10.0+cu128
Note: you may need to restart the kernel to use updated packages.


============================================================================
SECTION 2.6: DATA SPLITTING
============================================================================


SECTION 2.7: DATA AUGMENTATION
============================================================================

## Data Augmentation
The notebook combines online augmentation in training transforms with optional dataset-level preparation steps.
Training-time transforms include resized crops, color jitter, horizontal flips, and random erasing; evaluation uses test-time augmentation with five-crop plus horizontal flips.

============================================================================
SECTION 3: DATASET CLASS & DATALOADERS
============================================================================

============================================================================
SECTION 4: MODEL DEFINITIONS
============================================================================

## Model Architecture
This pipeline supports multiple backbones, with ArcFace-focused training primarily using InceptionResnetV1 variants and an EfficientNet embedding wrapper.
Backbone selection and training mode are controlled centrally in Config, enabling consistent comparison across models.


SECTION 5: TRAINING FUNCTIONS
============================================================================


SECTION 6: MAIN TRAINING LOOP
============================================================================

## Training Loop
Training combines transfer learning, ArcFace optimization, and early stopping to improve generalization on a compact dataset.
The loop includes staged optimization behavior (including epoch-6 optimizer rebuild), checkpointing best validation performance, and plotting learning curves for review.


SECTION 7: EVALUATION
============================================================================

## Evaluation
Evaluation reports accuracy, class-wise metrics, confusion matrix visualizations, and optional misclassified-sample inspection.
For ArcFace models, logits are reconstructed from saved arcface_eval metadata, and test-time augmentation averages five-crop plus flipped views.


SECTION 8: MAIN EXECUTION
============================================================================

## Final Results
The main execution stage trains configured models, evaluates each model, and exports a consolidated comparison table.
Primary output artifacts include best checkpoints, training/evaluation plots, and model_comparison.csv for reviewer-friendly summary analysis.


RUN TRAINING
============================================================================

**ResNet‑50 training failed here** because the class‑weights tensor was left on the CPU while the model was on the GPU.  
The corrected training cell (with the device fix) is provided below.


RESNET50 TRAINING (IMAGE‑CLASSIFICATION HEAD)
============================================================================
This cell trains an ImageNet‑pretrained ResNet‑50 with a classification head
(CrossEntropyLoss + label smoothing + class weights). It serves as a non‑ArcFace
baseline for comparison with the face‑specific models.

In [63]:
import shutil
import os

base_dir = "/kaggle/working"
zip_path = "/kaggle/working/project_outputs"

folders_to_zip = ["models", "plots", "results"]

# Create a temp directory to gather selected folders
temp_dir = "/kaggle/working/_temp_zip"
os.makedirs(temp_dir, exist_ok=True)

# Copy selected folders into temp dir
for folder in folders_to_zip:
    src = os.path.join(base_dir, folder)
    dst = os.path.join(temp_dir, folder)
    
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"✓ Added: {folder}")
    else:
        print(f"Skipped (not found): {folder}")

# Create zip
shutil.make_archive(zip_path, 'zip', temp_dir)

print(f"\nZip created: {zip_path}.zip")

# Optional: remove temp folder
shutil.rmtree(temp_dir)

✓ Added: models
✓ Added: plots
✓ Added: results

Zip created: /kaggle/working/project_outputs.zip


In [64]:
import shutil, os
from IPython.display import FileLink

raw_merged_dir = "data/raw_merged"
zip_base = "/kaggle/working/raw_merged"          # without .zip extension
zip_path = zip_base + ".zip"

if os.path.isdir(raw_merged_dir):
    # Create the zip archive
    shutil.make_archive(zip_base, 'zip', raw_merged_dir)
    print(f"✔ Archive created: {zip_path}")
    
    # Show a clickable download link
    display(FileLink(zip_path))
else:
    print("⚠️ 'data/raw_merged' not found. Run the merge cell first.")

✔ Archive created: /kaggle/working/raw_merged.zip


/kaggle/working/raw_merged.zip